## 6.Runnables + LCEL

### What is a Runnable?
A Runnable is simply a LangChain component that can take an input and produce an output.<BR>
<pre>
                  Input
                    ↓
                 Runnable
                    ↓
                  Output

              <u>Example:</u>
                  Prompt
                    ↓
                  Gemini
                    ↓
                  AI response
In this example Gemini model is a runnable
</pre>

### What is this '|' ?
<pre>
  prompt | model | parser

Take the output from the left component and pass it as input to the right component.

  prompt
    ↓
  model
    ↓
  parser

  It's basically a pipeline.
</pre>


### What is LCEL?

<pre>
LCEL stands for: LangChain Expression Language
LCEL is basically the syntax/style LangChain provides for composing Runnables into chains.

<u>ex:</u>
chain = prompt | model | parser
This is LCEL
</pre>

### Remember our Module 3 prompt template?

<pre>
Let's connect this to what you've already learned
</pre>

In [7]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-20b")

In [8]:
# prompt
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template("Explain {topic} in simple words.")

In [9]:
# parser
from langchain_core.output_parsers import StrOutputParser

parser=StrOutputParser()

In [10]:
chain=prompt | model | parser


In [11]:
res=chain.invoke({"topic":"Java threading concept"})
print(res)

## Java Threading – The Basics (in plain English)

### 1. What is a Thread?

Think of a **thread** as a *mini‑computer* that can run inside your program.  
- Your whole Java program is one “computer” (the *main thread*).  
- When you want to do several things at the same time – like download a file while still responding to user clicks – you give each job its own thread.

### 2. Why Use Threads?

| Problem | Thread‑based solution |
|---------|-----------------------|
| One job blocks everything else (e.g., reading a file) | Put the job in its own thread; the main thread keeps running. |
| Want to use multiple CPU cores | Run several threads in parallel; each can use a different core. |

### 3. Thread Life Cycle

```
NEW          (created but not started)
RUNNABLE      (ready to run or actually running)
BLOCKED      (waiting for a monitor lock)
WAITING      (waiting forever, e.g., Object.wait())
TIMED_WAITING (sleeping or waiting with a timeout)
TERMINATED   (finished)
```

### 4. How t

<pre>
{"topic": "Java threading concept"}
        ↓
      prompt
        ↓
"Explain Java threading concept in simple words."
        ↓
      model
        ↓
   AI response
        ↓
     parser
        ↓
   final string
</pre>

### RunnableLambda 
turn a normal Python function into a LangChain Runnable.

In [12]:
from langchain_core.runnables import RunnableLambda

def uppercase(text):
    return text.upper()

uppercase_runnable=RunnableLambda(uppercase)
result=uppercase_runnable.invoke("hello")
print(result)


HELLO


Now we can put it into a chain:

In [13]:
chain=prompt | model | parser | uppercase_runnable
res=chain.invoke({"topic":"Lambda function in javascript"})
print(res)

### WHAT IS A “LAMBDA” IN JAVASCRIPT?

IN JAVASCRIPT THE TERM *LAMBDA* IS USUALLY A NICKNAME FOR AN **ARROW FUNCTION**.  
THINK OF IT AS A TINY, ANONYMOUS FUNCTION THAT YOU CAN WRITE INLINE, RIGHT WHERE YOU NEED IT.

---

#### 1.  THE IDEA

- **ANONYMOUS** – IT DOESN’T NEED A NAME.  
- **SHORT** – THE SYNTAX IS COMPACT.  
- **INLINE** – YOU CAN DROP IT INTO AN ARGUMENT LIST OR A VARIABLE DECLARATION.

IT’S THE JAVASCRIPT EQUIVALENT OF THE “LAMBDA” YOU SEE IN PYTHON (`LAMBDA X: X+1`) OR OTHER LANGUAGES.

---

#### 2.  BASIC SYNTAX

```JS
// TRADITIONAL FUNCTION
FUNCTION ADD(A, B) {
  RETURN A + B;
}

// ARROW FUNCTION (LAMBDA)
CONST ADD = (A, B) => A + B;
```

*KEY POINTS:*

- `=>` IS THE ARROW THAT SEPARATES PARAMETERS FROM THE BODY.  
- IF YOU HAVE **ONE PARAMETER**, YOU CAN OMIT THE PARENTHESES:  
  `CONST DOUBLE = X => X * 2;`  
- IF THE BODY IS A SINGLE EXPRESSION, YOU CAN OMIT THE BRACES AND THE `RETURN` KEYWORD.

---

#### 3.  WHEN TO USE IT

| USE CASE | EXAMPLE |
|----------|--

### RunnablePassthrough
<pre>
It literally means:

"Just pass this through."
</pre>

In [14]:
from langchain_core.runnables import RunnablePassthrough

passthrough=RunnablePassthrough()
result=passthrough.invoke("Python")
print(result)

Python


### RunnableParallel
It sends the same input to multiple branches.

In [3]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda

parallel=RunnableParallel(
    original=RunnablePassthrough(),
    uppercase=RunnableLambda(lambda x:x.upper())
)
result=parallel.invoke("Hello bro")
print(result)

{'original': 'Hello bro', 'uppercase': 'HELLO BRO'}


### 🛠️ AI Content Generator Pipeline

In [4]:
from langchain_core.prompts import ChatPromptTemplate

title_prompt=ChatPromptTemplate.from_template(
    """
     You are an expert content writer.
     Generate a clear,interesting,and engaging title for the given topic.
     Topic: {topic}
     return only the title
    """
)

outline_prompt=ChatPromptTemplate.from_template(
    """
     You are an expert content writer.
     Create a clear and well-structured outline for an article.
     Topic: {topic}
     Title:{title}
     Include the main sections and important points.
     Return only the outline.
    """
)

content_prompt=ChatPromptTemplate.from_template(
    """
    You are an expert content writer.
    Write a detailed and easy-to-understand article using the information below.
    Topic: {topic}
    Title: {title}
    Outline:{outline}
    Write the complete content following the outline.
    """
)

summary_prompt = ChatPromptTemplate.from_template(
    """
    You are an expert summarizer.

    Create a concise summary of the following content.

    Content:
    {content}

    Return only the summary.
    """
)


<pre>
The parser converts:
   AIMessage → string
</pre>

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser=StrOutputParser()


In [10]:
title_chain=title_prompt | model | parser
title=title_chain.invoke({
    "topic":"Artificial Intelligence"
})
print(title)

AI Revolution: Redefining the Future


In [11]:
# add RunnableLambda

def lower_text(text):
    return text.lower()

In [13]:
# convert lower_text() into a runnable
from langchain_core.runnables import RunnableLambda

lower_text_runnable=RunnableLambda(lower_text)
title_chain=title_prompt | model | parser | lower_text_runnable
title=title_chain.invoke({
    "topic":"Artificial Intelligence"
})
print(title)

the ai revolution: transforming tomorrow, today


<pre>
Now we have a problem.

Our next prompt needs:

topic
title

But title_chain only gives us the title.

We need to preserve the original topic.

This is where:

RunnablePassthrough()

comes in.
</pre>

In [17]:
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

topic_title_chain=RunnableParallel(
    topic=RunnablePassthrough(),
    title=title_chain
)
res=topic_title_chain.invoke({
    "topic":"Artificial Intelligence"
})
print(res)

{'topic': {'topic': 'Artificial Intelligence'}, 'title': "ai unleashed: the future we can't ignore"}


In [18]:
# create outline chain

outline_chain=outline_prompt | model | parser
outline=outline_chain.invoke(res)
print(outline)

**Outline for “AI Unleashed: The Future We Can’t Ignore”**

1. **Introduction**
   - Hook: Rapid AI breakthroughs reshaping everyday life.
   - Thesis: AI’s transformative power, coupled with ethical and societal challenges, demands urgent attention.
   - Roadmap: Preview of key themes—technological advances, applications, impacts, risks, and policy responses.

2. **The Rise of Artificial Intelligence**
   - Historical milestones: From Turing to deep learning.
   - Core technologies: Neural networks, reinforcement learning, generative models.
   - Current scale: Data volumes, computing power, cloud infrastructure.

3. **Transformative Applications**
   - **Healthcare**: Diagnostics, personalized medicine, drug discovery.
   - **Transportation**: Autonomous vehicles, traffic optimization.
   - **Finance**: Algorithmic trading, fraud detection, credit scoring.
   - **Creative Industries**: AI‑generated art, music, content creation.
   - **Education**: Adaptive learning platforms, virtual

<pre>
Now we want:

  {
    "topic": "...",
    "title": "...",
    "outline": "..."
  }
</pre>

In [19]:
# We can use RunnableParallel
outline_full_chain = RunnableParallel(
    topic=RunnableLambda(lambda x: x["topic"]),
    title=RunnableLambda(lambda x: x["title"]),
    outline=outline_chain
)

complete_outline_chain=topic_title_chain | outline_full_chain


In [23]:
result=complete_outline_chain.invoke({
    "topic":"Artificial Intelligence"
})
print(result)

{'topic': {'topic': 'Artificial Intelligence'}, 'title': "ai unlocked: the revolution that's redefining reality", 'outline': "**AI Unlocked: The Revolution That's Redefining Reality**\n\n1. **Introduction**\n   - Hook: AI’s rapid ascent into everyday life\n   - Thesis: AI is not just a tool—it’s reshaping reality\n   - Overview of article structure\n\n2. **The Evolution of Artificial Intelligence**\n   - Early milestones: Turing, perceptrons, expert systems\n   - The deep‑learning breakthrough (2012 ImageNet)\n   - Current generation: GPT‑4, diffusion models, multimodal AI\n\n3. **Core Technologies Driving the Revolution**\n   - Neural networks & deep learning architectures\n   - Reinforcement learning & self‑improving agents\n   - Transfer learning & few‑shot learning\n   - Edge AI and hardware acceleration\n\n4. **AI Across Industries**\n   - Healthcare: diagnostics, personalized medicine, drug discovery\n   - Finance: algorithmic trading, fraud detection, credit scoring\n   - Manufa

In [24]:
# create content chain

content_chain=content_prompt | model | parser
content=content_chain.invoke(result)
print(content)

# AI Unlocked: The Revolution That’s Redefining Reality  

---

## 1. Introduction  

### Hook: AI’s Rapid Ascent Into Everyday Life  

If you’ve ever typed a query into Google, chatted with a customer‑service bot, or had a music playlist that “just knows” what you want, you’ve already lived in the AI‑powered world. Within the last decade, artificial intelligence has leapt from academic labs to the very fabric of our daily routines. From self‑driving cars on our roads to AI‑driven medical diagnostics in hospitals, the presence of machine intelligence is no longer a futuristic concept— it’s here, humming quietly in the background of our lives.

### Thesis: AI Is Not Just a Tool—it’s Reshaping Reality  

While many people think of AI as a set of tools—chatbots, recommendation engines, image recognizers— the truth is deeper. AI is redefining *how* we perceive and interact with reality. It’s changing the rules of work, the nature of creativity, the way we learn, and even how we govern ours

In [25]:
# add content to our data

content_full_chain = RunnableParallel(
    topic=RunnableLambda(lambda x: x["topic"]),
    title=RunnableLambda(lambda x: x["title"]),
    outline=RunnableLambda(lambda x: x["outline"]),
    content=content_chain
)

In [26]:
content_full_chain = RunnableParallel(
    topic=RunnableLambda(lambda x: x["topic"]),
    title=RunnableLambda(lambda x: x["title"]),
    outline=RunnableLambda(lambda x: x["outline"]),
    content=content_chain
)

In [28]:
complete_content_chain = complete_outline_chain | content_full_chain

In [29]:
result = complete_content_chain.invoke({
    "topic": "Artificial Intelligence"
})

print(result)

{'topic': {'topic': 'Artificial Intelligence'}, 'title': 'unleashing tomorrow: the power and promise of artificial intelligence', 'outline': '**Outline – “Unleashing Tomorrow: The Power and Promise of Artificial Intelligence”**\n\nI. **Introduction**\n   - Hook: AI as the next industrial revolution  \n   - Purpose of the article: explore capabilities, benefits, and future impact  \n   - Thesis statement: AI’s transformative power is matched by its promise to solve global challenges\n\nII. **Foundations of Artificial Intelligence**\n   - Definition and scope of AI  \n   - Core subfields: machine learning, deep learning, natural language processing, computer vision, robotics  \n   - Key technologies: neural networks, reinforcement learning, generative models, edge AI  \n\nIII. **The Power of AI – Current Applications**\n   1. **Healthcare**\n      - Diagnostics (image analysis, predictive modeling)  \n      - Personalized medicine & drug discovery  \n      - Remote monitoring & telehealt

<pre>
Now we have:

topic
title
outline
content
</pre>

In [30]:
summary_chain = summary_prompt | model | parser
summary = summary_chain.invoke(result)

print(summary)

**Summary**

Artificial Intelligence is no longer a speculative concept but a driving force behind today’s most transformative technologies—from medical diagnostics and autonomous vehicles to climate modeling and creative media. The article outlines AI’s foundations, including machine learning, deep learning, NLP, computer vision, and robotics, and highlights key technologies such as neural networks, reinforcement learning, generative models, and edge AI.

Current applications span healthcare (diagnostics, personalized medicine, telehealth), manufacturing (predictive maintenance, cobots, quality control), finance (algorithmic trading, risk assessment, fraud detection), transportation (self‑driving cars, smart logistics, transit forecasting), and creative industries (generative art, automated production, personalized content). Looking ahead, AI promises to tackle global challenges like climate change, renewable energy optimization, food security, and pandemic prediction, while also enab

In [31]:
final_chain = RunnableParallel(
    topic=RunnableLambda(lambda x: x["topic"]),
    title=RunnableLambda(lambda x: x["title"]),
    outline=RunnableLambda(lambda x: x["outline"]),
    content=RunnableLambda(lambda x: x["content"]),
    summary=summary_chain
)

In [32]:
complete_final_chain = complete_content_chain | final_chain
result = complete_final_chain.invoke({
    "topic": "Artificial Intelligence"
})

In [34]:
print("========== TITLE ==========")
print(result["title"])

print("\n========== OUTLINE ==========")
print(result["outline"])

print("\n========== CONTENT ==========")
print(result["content"])

print("\n========== SUMMARY ==========")
print(result["summary"])

========== TITLE ==========
ai unleashed: how machines are reshaping humanity

========== OUTLINE ==========
**Outline: “AI Unleashed: How Machines Are Reshaping Humanity”**

1. **Introduction**
   - Hook: A striking statistic or anecdote illustrating AI’s rapid rise.
   - Thesis statement: AI is not just a technological trend—it’s a transformative force reshaping every facet of human life.
   - Overview of article structure.

2. **Historical Context**
   - Early milestones: Turing, perceptrons, expert systems.
   - The AI winter(s): causes and lessons learned.
   - Resurgence: deep learning breakthroughs (AlexNet, GPT, AlphaGo).

3. **Current Capabilities of AI**
   - **Natural Language Processing** – ChatGPT, voice assistants, translation.
   - **Computer Vision** – facial recognition, medical imaging, autonomous vehicles.
   - **Reinforcement Learning & Robotics** – game AI, industrial automation.
   - **Generative Models** – art, music, synthetic data.
   - **AI in Decision‑Making*